# MixesDB search → tracklists

Enter a search word (e.g. `hawtin`). This notebook:
1. Searches MixesDB for mixes whose **title** contains the word (via `mixesdb_scraper.py`)
2. Fetches the tracklist for each matching mix (via `mixesdb_tracklist.py`)
3. Prints the tracklist for every mix that has one, and skips those without
4. Optionally saves everything to a single CSV

In [7]:
import sys, os
sys.path.insert(0, os.getcwd())

from mixesdb_scraper import search
from mixesdb_tracklist import get_tracklist
import pandas as pd

## 1. Enter the search word

In [48]:
query = 'DVS1'  # search query; change to whatever you want
max_mixes = 2000  # how many matching mixes to fetch tracklists for; raise if you want more

print(f"Searching MixesDB for '{query}'...")

Searching MixesDB for 'DVS1'...


In [49]:
query

'DVS1'

## 2. Run the search (title must contain the word)

In [50]:
results = search(query, limit=max_mixes, strict=True)

print(f"Found {len(results)} mix(es) with '{query}' in the title.\n")
pd.DataFrame(results)[["title", "url"]] if results else print("No matching mixes found.")

Found 44 mix(es) with 'DVS1' in the title.



,title,url
0,"2012-09-29 - DVS1 @ Klubnacht, Berghain, Berli...",https://www.mixesdb.com/w/2012-09-29_-_DVS1_@_...
1,"2013-03-30 - DVS1 @ Foundry, BLK BOX, Toronto,...",https://www.mixesdb.com/w/2013-03-30_-_DVS1_@_...
2,"2012-10-12 - DVS1 @ Torque 001, eleven, Tokyo",https://www.mixesdb.com/w/2012-10-12_-_DVS1_@_...
3,"2010-03-20 - DVS1 @ Black, First Avenue/VIP Ro...",https://www.mixesdb.com/w/2010-03-20_-_DVS1_@_...
4,"2016-06-17 - DVS1 b2b Rødhåd @ Sonar By Night,...",https://www.mixesdb.com/w/2016-06-17_-_DVS1_b2...
5,"2009-11-28 - DVS1 @ 33rd BDay, Black, Minneapolis",https://www.mixesdb.com/w/2009-11-28_-_DVS1_@_...
6,"2009-07-18 - DVS1 @ Future Classic Part II, Hu...",https://www.mixesdb.com/w/2009-07-18_-_DVS1_@_...
7,"2010-03-06 - DVS1 @ Future Classic Part VI, Hu...",https://www.mixesdb.com/w/2010-03-06_-_DVS1_@_...
8,2017-12-08 - DVS1 - fabric 96,https://www.mixesdb.com/w/2017-12-08_-_DVS1_-_...
9,"2012 - DVS1 @ Unknown Gig, Minneapolis (fabric...",https://www.mixesdb.com/w/2012_-_DVS1_@_Unknow...


## 3. Fetch and display the tracklist for each mix

In [51]:
all_tracks = []  # flat rows for the CSV export: mix_title, mix_url, part, position, timestamp, artist, title, raw

for r in results:
    print(f"\n{'='*80}\n{r['title']}\n{r['url']}\n{'='*80}")

    try:
        tracks = get_tracklist(r["url"])
    except Exception as e:
        print(f"  Could not fetch tracklist: {e}")
        continue

    if not tracks:
        print("  No tracklist found on this page.")
        continue

    current_part = None
    for t in tracks:
        if t["part"] != current_part:
            current_part = t["part"]
            if current_part:
                print(f"\n-- {current_part} --")
        ts = f"[{t['timestamp']}] " if t["timestamp"] else ""
        line = f"{t['artist']} - {t['title']}" if t["artist"] else t["raw"]
        print(f"{t['position']:>3}. {ts}{line}")

        all_tracks.append({
            "mix_title": r["title"],
            "mix_url": r["url"],
            **t,
        })

# print(f"\n\nDone. Collected {len(all_tracks)} tracks across {len(results)} mix(es).")


2012-09-29 - DVS1 @ Klubnacht, Berghain, Berlin (Slam Radio 013 / Invite's Choice 100)
https://www.mixesdb.com/w/2012-09-29_-_DVS1_@_Klubnacht,_Berghain,_Berlin_(Slam_Radio_013_/_Invite's_Choice_100)

-- Slam Radio 013 (2012-12-27) --
  1. [00] Felix Da Housecat - Metropolis (Radikal Victory Edit)
  2. [03] Club MCM - Club MCM
  3. [03] Lighter Thief - Zippo King [Ifach - 019]
  4. [06] ?
  5. [09] ?
  6. [12] 2000 and One - Funk That [Wasabi - COMP 081]
  7. [15] ?
  8. [17] Surgeon - Atol
  9. [21] DJ Bone - We Control The Beat [Subject Detroit - SUB-030]
 10. [26] DVS1 - Polyphonic Love [Transmat - MS 98]
 11. [32] Untold - Motion The Dance [Hemlock - HEK016i]
 12. [37] ?
 13. [41] Operator - Hermes [Gnosis - 001]
 14. [44] Developer - Formu [Gynoid - 10]
 15. [47] Magnus - Act One [Magnus - 001]
 16. [49] ?
 17. [52] Falko Brocksieper - Excellent Upload (Conforce Reconfiguration) [Sub Static - SUS 078]
 18. [55] Architectural - Looking Ahead [Semantica - 40]

-- Invite's Choice 10

## 4. (optional) Save all tracks to CSV

In [52]:
if all_tracks:
    out_path = f"{query.replace(' ', '_')}_tracklists.csv"
    df = pd.DataFrame(all_tracks)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {out_path}")
else:
    print("Nothing to save — no tracklists were found.")

Saved 442 rows to DVS1_tracklists.csv
